# LightGBM / XGBoost — Tune & deploy a 24h-ahead vol forecaster

## Project goal

Train a LightGBM regressor that predicts BTC's next-24h realised volatility. Cover early stopping, monotonic constraints, custom MAE objective, quantile regression for prediction intervals, and feature importance comparison (gain vs permutation). Save as a deployable joblib bundle.


## Why this exercises the cheatsheet

Vol forecasting is the canonical regression problem on this dataset and every gradient-boosting feature has a natural role: early stopping for the regression budget, monotonic constraints to encode 'past vol → future vol' domain knowledge, custom objective for asymmetric loss, quantile regression for risk-aware downstream consumers.


## Sub-task 1: Features and target

Build past-only features: 1h/4h/24h log returns, 24h/168h rolling realised vol, hour-of-day, ATR, RSI. Target: realised vol over the next 24 hours, computed strictly forward from each row. Train/val/test chronological 70/15/15.

**Patterns this forces:**

- feature engineering with shift before rolling
- forward target via `rolling(24).std().shift(-24)`
- chronological split with purge horizon = 24


In [ ]:
# Your answer here

## Sub-task 2: Baseline LightGBM with early stopping

Fit `lgb.LGBMRegressor(objective='regression_l1', ...)` on train with `eval_set=[(X_val, y_val)]` and `lgb.early_stopping(20)` callback. Report the iteration where it stopped and val MAE.

**Patterns this forces:**

- `callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)]`
- `m.best_iteration_` after fit
- `objective='regression_l1'` for MAE


In [ ]:
# Your answer here

## Sub-task 3: Add a monotonic constraint

Domain knowledge: higher trailing 24h vol should imply higher (or equal) predicted vol. Refit with `monotone_constraints=` (1 for `vol_24h_trailing`, 0 for everything else). Compare val MAE to the unconstrained model. **Comment**: did the constraint help or hurt?

**Patterns this forces:**

- `monotone_constraints=[0, 1, 0, 0, ...]`
- compare val MAE constrained vs unconstrained


In [ ]:
# Your answer here

## Sub-task 4: Custom MAE objective (advanced)

Implement an asymmetric custom objective: penalise UNDER-prediction of vol twice as heavily as over-prediction (because under-predicting risk is worse than over-predicting). Pass it as `objective=...`.

**Patterns this forces:**

- function returning `(grad, hess)`
- asymmetric grad: `np.where(y_pred < y_true, -2.0, 1.0)`
- constant small hess (~0.01) to avoid divide-by-zero


In [ ]:
# Your answer here

## Sub-task 5: Quantile regression for [p10, p90]

Train two quantile models: `objective='quantile', alpha=0.1` and `alpha=0.9`. Predict on test. Compute empirical coverage of [p10, p90]: the fraction of test points where p10 ≤ true ≤ p90. Should be ~0.80.

**Patterns this forces:**

- `lgb.LGBMRegressor(objective='quantile', alpha=...)`
- two models, two predictions
- coverage = `((y_test >= p10) & (y_test <= p90)).mean()`


In [ ]:
# Your answer here

## Sub-task 6: Feature importance: gain vs permutation

Compute (a) `model.booster_.feature_importance(importance_type='gain')` and (b) `permutation_importance(model, X_val, y_val, n_repeats=5)`. Build a DataFrame with both rankings. Comment on disagreement: any feature ranked high by gain but low by permutation? That's a sign of high-cardinality feature with many splits but little generalisation.

**Patterns this forces:**

- `model.booster_.feature_importance(importance_type='gain')`
- `from sklearn.inspection import permutation_importance`
- side-by-side ranking comparison


In [ ]:
# Your answer here

## Sub-task 7: Save the bundle

Save the chosen model + feature_names + threshold + trained_through timestamp as a `joblib` bundle. Verify by loading and predicting on one test row.

**Patterns this forces:**

- `joblib.dump({'model': ..., 'feature_names': ..., ...}, path)`
- bundle includes everything inference needs
- load + smoke-test in the same cell


In [ ]:
# Your answer here

## What success looks like

- One LightGBM model with monotonic constraint, fit via early stopping.
- A quantile pair (p10, p90) achieving empirical coverage in [0.75, 0.85] on test.
- A feature-importance comparison table that shows gain ranking ≠ permutation ranking somewhere.
- A `joblib` artifact under 5 MB with everything an inference service needs.
- The custom-objective experiment either improved val MAE or you've documented why it didn't.
